# Stage 2 – Anomaly Detection + Full Pipeline Visualization

This notebook:
1. **Trains a Stage 2 anomaly classifier** (Faster R-CNN) on the DENTEX-2023 dataset.
2. **Reloads Stage 1** Mask R-CNN teeth segmentation weights (uploaded dataset).
3. **Runs the full combined pipeline** per image: segment → crop each tooth → classify anomaly.
4. **Renders a 4-panel figure** identical to Stage 1 layout, with quadrant lines, tooth numbers,
   and anomalous teeth highlighted in red with a detailed anomaly report panel.

**Dataset paths (Kaggle)**
| Dataset | Kaggle path |
|---|---|
| Stage 1 weights (uploaded) | `/kaggle/input/maskrcnn-teeth-stage1-weights/` |
| DENTEX-2023 training data | `/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/` |


In [ ]:
print("=" * 60)
print("CELL 1: Imports, paths & config")
print("=" * 60)
# ================== CELL 1: SETUP & CONFIG ==================

import os, json, random, copy, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np
import cv2
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import nms
import torchvision.transforms.functional as TF

print("[1/4] All imports OK")

print(f"PyTorch:     {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")

# ── Paths ──────────────────────────────────────────────────────────────────
STAGE1_WEIGHTS_DIR   = "/kaggle/input/maskrcnn-teeth-stage1-weights"
STAGE1_BEST_WEIGHTS  = os.path.join(STAGE1_WEIGHTS_DIR, "maskrcnn_teeth_best.pth")
STAGE1_FINAL_WEIGHTS = os.path.join(STAGE1_WEIGHTS_DIR, "maskrcnn_teeth_final.pth")

DENTEX_ROOT = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023"

# Robustly find the image/label directories – DENTEX-2023 zip structure varies
def _find_dentex_dirs(root):
    candidates = [
        ("quadrant_enumeration_disease/train/images",
         "quadrant_enumeration_disease/train/labels"),
        ("train/xrays", "train/labels"),
        ("xrays", "labels"),
        ("training_data", "training_data"),
    ]
    for img_sub, lbl_sub in candidates:
        ip = os.path.join(root, img_sub)
        lp = os.path.join(root, lbl_sub)
        if os.path.isdir(ip):
            print(f"  ✓ Found DENTEX images at: {ip}")
            print(f"  ✓ Labels at:              {lp if os.path.isdir(lp) else '(not found – will skip)'}")
            return ip, lp if os.path.isdir(lp) else ip
    # fallback: walk and find a dir with .png / .jpg files
    for dirpath, _, files in os.walk(root):
        if any(f.endswith((".png",".jpg")) for f in files):
            print(f"  ✓ Fallback DENTEX images: {dirpath}")
            return dirpath, dirpath
    raise FileNotFoundError(f"Could not locate DENTEX images under {root}")

DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR = _find_dentex_dirs(DENTEX_ROOT)

STAGE2_BEST      = "/kaggle/working/stage2_anomaly_best.pth"
STAGE2_FINAL     = "/kaggle/working/stage2_anomaly_final.pth"
OUT_DIR          = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Anomaly classes (DENTEX-2023 task 3) ──────────────────────────────────
ANOMALY_CLASSES = {
    0: "background",
    1: "Caries",
    2: "Deep Caries",
    3: "Periapical Lesion",
    4: "Impacted Tooth",
}
NUM_ANOMALY_CLASSES = len(ANOMALY_CLASSES)   # 5  (background + 4 disease)

CONFIG2 = {
    "batch_size":     2,
    "lr":             1e-4,
    "weight_decay":   1e-5,
    "epochs":         30,
    "conf_threshold": 0.45,
    "nms_iou":        0.3,
    "train_ratio":    0.75,
    "val_ratio":      0.15,
    "num_workers":    2,
    "crop_pad":       12,
    "min_crop_size":  24,
}

CONFIG1 = {
    "conf_threshold": 0.6,
    "nms_iou":        0.3,
    "padding":        20,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[2/4] Device: {device}")
print(f"\n✓ Device: {device}")
for p, label in [
    (STAGE1_BEST_WEIGHTS,  "Stage 1 best weights"),
    (DENTEX_ROOT,          "DENTEX-2023 root"),
    (DENTEX_TRAIN_IMG,     "DENTEX train images"),
    (DENTEX_LABEL_DIR,     "DENTEX labels"),
]:
    status = "✓" if os.path.exists(p) else "✗ NOT FOUND"
    print(f"  {status}  {label}")


print("[3/4] DENTEX dirs resolved:")
print(f"      images: {DENTEX_TRAIN_IMG}")
print(f"      labels: {DENTEX_LABEL_DIR}")
print("[4/4] Config ready")
print()
print("CELL 1 COMPLETE ✓")

In [ ]:
print("=" * 60)
print("CELL 2: DentexAnomalyDataset class")
print("=" * 60)

# ================== CELL 2: DENTEX-2023 DATASET ==================
# Supports YOLO .txt labels AND Supervisely / COCO-style JSON annotations.
# category_id: 1=Caries, 2=Deep Caries, 3=Periapical Lesion, 4=Impacted Tooth

class DentexAnomalyDataset(Dataset):
    CATEGORY_MAP = {1: 1, 2: 2, 3: 3, 4: 4}   # raw id → model class id

    def __init__(self, img_dir, label_dir, file_list=None):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        all_imgs = (
            sorted(glob.glob(os.path.join(img_dir,"*.png")) +
                   glob.glob(os.path.join(img_dir,"*.jpg")))
            if file_list is None
            else [os.path.join(img_dir, f) for f in file_list]
        )
        self.samples = []
        for img_path in all_imgs:
            base   = os.path.splitext(os.path.basename(img_path))[0]
            lbl_t  = os.path.join(label_dir, base + ".txt")
            lbl_j  = os.path.join(label_dir, base + ".json")
            if os.path.exists(lbl_t):
                self.samples.append((img_path, lbl_t, "yolo"))
            elif os.path.exists(lbl_j):
                self.samples.append((img_path, lbl_j, "json"))
            # images without labels are still kept (they produce empty targets)
            # so the model sees "no disease" examples
            else:
                self.samples.append((img_path, None, None))
        print(f"✓ DentexAnomalyDataset: {len(self.samples)} images  "
              f"({sum(1 for _,l,_ in self.samples if l)} labelled)")

    def __len__(self): return len(self.samples)

    def _load_yolo(self, path, w, h):
        boxes, labels = [], []
        for line in open(path):
            p = line.strip().split()
            if len(p) < 5: continue
            cid = int(p[0]) + 1                         # 0-indexed → 1-indexed
            if cid not in self.CATEGORY_MAP: continue
            cx, cy, bw, bh = map(float, p[1:5])
            x1 = (cx - bw/2) * w;  y1 = (cy - bh/2) * h
            x2 = (cx + bw/2) * w;  y2 = (cy + bh/2) * h
            if x2-x1 < 2 or y2-y1 < 2: continue
            boxes.append([x1,y1,x2,y2]);  labels.append(self.CATEGORY_MAP[cid])
        return boxes, labels

    def _load_json(self, path, w, h):
        boxes, labels = [], []
        data = json.load(open(path))
        anns = data.get("annotations", data.get("objects", []))
        for ann in anns:
            cid = ann.get("category_id", ann.get("label_id", -1))
            if cid not in self.CATEGORY_MAP: continue
            if "bbox" in ann:
                bx,by,bw,bh = ann["bbox"];  x1,y1,x2,y2 = bx,by,bx+bw,by+bh
            elif "points" in ann:
                pts = np.array(ann["points"]["exterior"])
                x1,y1,x2,y2 = pts[:,0].min(),pts[:,1].min(),pts[:,0].max(),pts[:,1].max()
            else:
                continue
            if x2-x1 < 2 or y2-y1 < 2: continue
            boxes.append([x1,y1,x2,y2]);  labels.append(self.CATEGORY_MAP[cid])
        return boxes, labels

    def __getitem__(self, idx):
        img_path, lbl_path, fmt = self.samples[idx]
        img  = Image.open(img_path).convert("RGB")
        w, h = img.size
        if lbl_path and fmt == "yolo":
            boxes, labels = self._load_yolo(lbl_path, w, h)
        elif lbl_path and fmt == "json":
            boxes, labels = self._load_json(lbl_path, w, h)
        else:
            boxes, labels = [], []
        t = TF.to_tensor(img)
        if boxes:
            target = {"boxes":    torch.tensor(boxes,  dtype=torch.float32),
                      "labels":   torch.tensor(labels, dtype=torch.int64),
                      "image_id": torch.tensor([idx])}
        else:
            target = {"boxes":    torch.zeros((0,4),   dtype=torch.float32),
                      "labels":   torch.zeros((0,),    dtype=torch.int64),
                      "image_id": torch.tensor([idx])}
        return t, target

print("  Class defined OK")
print()
print("CELL 2 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 3: Train/Val/Test split & DataLoaders")
print("=" * 60)

# ================== CELL 3: SPLIT & DATALOADERS ==================

all_imgs = sorted([os.path.basename(p) for p in
    glob.glob(os.path.join(DENTEX_TRAIN_IMG,"*.png")) +
    glob.glob(os.path.join(DENTEX_TRAIN_IMG,"*.jpg"))])

random.seed(42); random.shuffle(all_imgs)
n_total = len(all_imgs)
n_train = int(n_total * CONFIG2["train_ratio"])
n_val   = int(n_total * CONFIG2["val_ratio"])
train_files = all_imgs[:n_train]
val_files   = all_imgs[n_train:n_train+n_val]
test_files  = all_imgs[n_train+n_val:]
print(f"Total: {n_total}  Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}")

train_ds = DentexAnomalyDataset(DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR, file_list=train_files)
val_ds   = DentexAnomalyDataset(DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR, file_list=val_files)
test_ds  = DentexAnomalyDataset(DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR, file_list=test_files)

def collate_fn(batch): return tuple(zip(*batch))

train_loader2 = DataLoader(train_ds, batch_size=CONFIG2["batch_size"], shuffle=True,
                            collate_fn=collate_fn, num_workers=CONFIG2["num_workers"],
                            pin_memory=True)
val_loader2   = DataLoader(val_ds,   batch_size=CONFIG2["batch_size"], shuffle=False,
                            collate_fn=collate_fn, num_workers=CONFIG2["num_workers"])
test_loader2  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                            collate_fn=collate_fn, num_workers=CONFIG2["num_workers"])
print(f"✓ Train batches: {len(train_loader2)}  Val batches: {len(val_loader2)}")
print(f"  Test batches: {len(test_loader2)}")
print()
print("CELL 3 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 4: Stage 2 Faster R-CNN model")
print("=" * 60)

# ================== CELL 4: STAGE 2 MODEL (FASTER R-CNN) ==================
# Faster R-CNN (no mask head) – DENTEX-2023 provides bounding-box labels only.

def get_anomaly_detector(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    return model

stage2_model = get_anomaly_detector(NUM_ANOMALY_CLASSES).to(device)
optimizer2   = torch.optim.AdamW(stage2_model.parameters(),
                                  lr=CONFIG2["lr"],
                                  weight_decay=CONFIG2["weight_decay"])
scheduler2   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=CONFIG2["epochs"])
print(f"✓ Stage 2 Faster R-CNN ready on {device}")
print(f"  Classes ({NUM_ANOMALY_CLASSES}): {list(ANOMALY_CLASSES.values())}")
total_params = sum(p.numel() for p in stage2_model.parameters())
train_params = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
print(f"  Total params: {total_params:,}  |  Trainable: {train_params:,}")
print(f"  Optimizer: AdamW  lr={CONFIG2['lr']}  wd={CONFIG2['weight_decay']}")
print()
print("CELL 4 COMPLETE ✓")


In [ ]:
# ================== CELL 5: TRAINING STAGE 2 ==================

print("=" * 60)
print(f"CELL 5: Training  ({CONFIG2['epochs']} epochs, device={device})")
print("=" * 60)

best_val2 = float("inf")
train_losses2, val_losses2 = [], []

for epoch in range(CONFIG2["epochs"]):
    # ── TRAIN ─────────────────────────────────────────────────────────────
    stage2_model.train()
    ep_tr = 0.0;  n_tr = 0
    for imgs, tgts in train_loader2:
        valid = [(i,t) for i,t in zip(imgs,tgts) if t["boxes"].shape[0] > 0]
        if not valid: continue
        im, tg = zip(*valid)
        im = [x.to(device) for x in im]
        tg = [{k: v.to(device) for k,v in t.items()} for t in tg]
        loss_dict = stage2_model(im, tg)
        loss = sum(loss_dict.values())
        optimizer2.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(stage2_model.parameters(), 5.0)
        optimizer2.step()
        ep_tr += loss.item();  n_tr += 1
    avg_tr = ep_tr / max(1, n_tr)
    train_losses2.append(avg_tr)

    # ── VALIDATION (in train mode to get loss) ────────────────────────────
    stage2_model.train()
    ep_vl = 0.0;  n_vl = 0
    with torch.no_grad():
        for imgs, tgts in val_loader2:
            valid = [(i,t) for i,t in zip(imgs,tgts) if t["boxes"].shape[0] > 0]
            if not valid: continue
            im, tg = zip(*valid)
            im = [x.to(device) for x in im]
            tg = [{k: v.to(device) for k,v in t.items()} for t in tg]
            ep_vl += sum(stage2_model(im, tg).values()).item();  n_vl += 1
    avg_vl = ep_vl / max(1, n_vl)
    val_losses2.append(avg_vl)
    scheduler2.step()

    marker = "  ★ BEST" if avg_vl < best_val2 else ""
    print(f"  Epoch {epoch+1:03d}/{CONFIG2['epochs']}  Train: {avg_tr:.4f}  Val: {avg_vl:.4f}{marker}")

    if avg_vl < best_val2:
        best_val2 = avg_vl
        torch.save({
            "epoch":               epoch + 1,
            "model_state_dict":    stage2_model.state_dict(),
            "optimizer_state_dict":optimizer2.state_dict(),
            "val_loss":            best_val2,
            "config":              CONFIG2,
            "anomaly_classes":     ANOMALY_CLASSES,
        }, STAGE2_BEST)
        print(f"  ✓ Best model saved  (val_loss={best_val2:.4f})")

torch.save(stage2_model.state_dict(), STAGE2_FINAL)
print("\n✓ Stage 2 training complete!")

# ── Training curves ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,4), facecolor="#1a1a2e")
ax.set_facecolor("#0d0d1a")
ax.plot(train_losses2, label="Train", color="#e74c3c", linewidth=2)
ax.plot(val_losses2,   label="Val",   color="#3498db", linewidth=2)
ax.set_title("Stage 2 – Training Curves", color="white", fontsize=14)
ax.set_xlabel("Epoch", color="#aaa"); ax.set_ylabel("Loss", color="#aaa")
ax.tick_params(colors="#aaa"); ax.legend(facecolor="#1a1a2e", labelcolor="white")
for sp in ax.spines.values(): sp.set_color("#444")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "stage2_curves.png"), dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("✓ Saved stage2_curves.png")
print()
print("CELL 5 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 6: Reload best Stage 2 weights")
print("=" * 60)

if not os.path.exists(STAGE2_BEST):
    raise FileNotFoundError(f"Checkpoint not found: {STAGE2_BEST} — did Cell 5 run?")

# ================== CELL 6: RELOAD BEST STAGE 2 WEIGHTS ==================

ckpt2 = torch.load(STAGE2_BEST, map_location=device)
stage2_model.load_state_dict(ckpt2["model_state_dict"])
stage2_model.eval()
print(f"✓ Stage 2 best weights reloaded  "
      f"(epoch={ckpt2['epoch']}, val_loss={ckpt2['val_loss']:.4f})")
print()
print("CELL 6 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 7: Evaluation — confusion matrix & report")
print("=" * 60)
print(f"  Running on {len(test_loader2)} test images...")

# ================== CELL 7: EVALUATION – CONFUSION MATRIX & REPORT ==================

all_true, all_pred = [], []
stage2_model.eval()

with torch.no_grad():
    for imgs, tgts in test_loader2:
        img  = imgs[0].to(device)
        tgt  = tgts[0]
        pred = stage2_model([img])[0]

        gt_labels   = tgt["labels"].numpy().tolist()
        pred_scores = pred["scores"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        pred_boxes  = pred["boxes"].cpu().numpy()
        gt_boxes    = tgt["boxes"].numpy()

        # Greedy IoU match (0.5 threshold) to pair GT ↔ pred
        matched_pred = set()
        for j, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
            best_iou, best_k = 0.0, -1
            for k, (pb, ps) in enumerate(zip(pred_boxes, pred_scores)):
                if k in matched_pred or ps < CONFIG2["conf_threshold"]: continue
                ix1,iy1 = max(gb[0],pb[0]),max(gb[1],pb[1])
                ix2,iy2 = min(gb[2],pb[2]),min(gb[3],pb[3])
                inter   = max(0,ix2-ix1)*max(0,iy2-iy1)
                union   = ((gb[2]-gb[0])*(gb[3]-gb[1]) +
                           (pb[2]-pb[0])*(pb[3]-pb[1]) - inter)
                iou     = inter/union if union>0 else 0.0
                if iou > best_iou: best_iou, best_k = iou, k
            if best_iou >= 0.5 and best_k >= 0:
                all_true.append(gl)
                all_pred.append(int(pred_labels[best_k]))
                matched_pred.add(best_k)
            else:
                all_true.append(gl);  all_pred.append(0)   # FN → predict background

labels_present = sorted(set(all_true + all_pred))
class_names    = [ANOMALY_CLASSES.get(l, f"class_{l}") for l in labels_present]

if len(all_true) > 0:
    cm = confusion_matrix(all_true, all_pred, labels=labels_present)
    fig, ax = plt.subplots(figsize=(8,6), facecolor="#1a1a2e")
    ax.set_facecolor("#0d0d1a")
    sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor="#333")
    ax.set_xlabel("Predicted", color="white", fontsize=12)
    ax.set_ylabel("Actual",    color="white", fontsize=12)
    ax.set_title("Stage 2 – Confusion Matrix (test set)", color="white", fontsize=14)
    ax.tick_params(colors="white")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR,"stage2_confusion_matrix.png"), dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print("✓ Saved stage2_confusion_matrix.png")

    report = classification_report(all_true, all_pred,
                                    labels=labels_present,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
    with open(os.path.join(OUT_DIR,"stage2_report.json"),"w") as f:
        json.dump(report, f, indent=2)
    print("✓ Saved stage2_report.json")
    print(classification_report(all_true, all_pred,
                                 labels=labels_present,
                                 target_names=class_names,
                                 zero_division=0))
else:
    print("⚠ No matched predictions to evaluate – test set may have no labelled anomalies.")
print()
print("CELL 7 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 8: Load Stage 1 Mask R-CNN + helper functions")
print("=" * 60)
print("  Loading Stage 1 backbone (weights=DEFAULT)...")

# ================== CELL 8: STAGE 1 HELPERS ==================
# Load Stage 1 Mask R-CNN weights from the uploaded Kaggle dataset.

def get_stage1_model(num_classes=33):
    m = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
    in_f  = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
    in_fm = m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor = MaskRCNNPredictor(in_fm, 256, num_classes)
    return m

stage1_model = get_stage1_model().to(device)
if os.path.exists(STAGE1_BEST_WEIGHTS):
    ck1 = torch.load(STAGE1_BEST_WEIGHTS, map_location=device)
    stage1_model.load_state_dict(ck1.get("model_state_dict", ck1))
    print(f"✓ Stage 1 best weights loaded  "
          f"(epoch={ck1.get('epoch','?')}, val_loss={ck1.get('val_loss','?')})")
elif os.path.exists(STAGE1_FINAL_WEIGHTS):
    stage1_model.load_state_dict(torch.load(STAGE1_FINAL_WEIGHTS, map_location=device))
    print("✓ Stage 1 final weights loaded")
else:
    print("⚠ No Stage 1 weights found – pipeline will not work correctly!")
stage1_model.eval()

# ── Stage 1 inference helpers ──────────────────────────────────────────────
def _nms_pred(pred, iou=CONFIG1["nms_iou"]):
    if len(pred["boxes"]) == 0: return pred
    keep = nms(pred["boxes"], pred["scores"], iou)
    return {k: v[keep] for k,v in pred.items()}

def run_stage1(img_tensor):
    with torch.no_grad():
        raw = stage1_model(img_tensor.to(device).unsqueeze(0))[0]
    keep = raw["scores"] >= CONFIG1["conf_threshold"]
    filt = _nms_pred({k: v[keep] for k,v in raw.items()})
    return {k: filt[k].cpu().numpy() for k in ["boxes","labels","masks","scores"]}

def crop_to_teeth(img_np, preds, pad=CONFIG1["padding"]):
    b = preds["boxes"]
    if len(b) == 0: return img_np, None
    x1 = max(0, int(b[:,0].min()) - pad)
    y1 = max(0, int(b[:,1].min()) - pad)
    x2 = min(img_np.shape[1], int(b[:,2].max()) + pad)
    y2 = min(img_np.shape[0], int(b[:,3].max()) + pad)
    return img_np[y1:y2, x1:x2], (x1, y1, x2, y2)

def color_teeth_fn(image, preds, crop_box=None):
    colored = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB) if image.ndim==2 else image.copy()
    palette = [
        [255,60,60],[60,255,60],[60,60,255],[255,220,0],[220,0,255],[0,220,255],
        [255,140,0],[140,0,255],[255,0,140],[0,255,140],[140,255,0],[0,140,255],
    ]
    ox, oy = (crop_box[0], crop_box[1]) if crop_box else (0, 0)
    ch, cw = colored.shape[:2]
    for i, (mask, box) in enumerate(zip(preds["masks"], preds["boxes"])):
        mb = (mask[0] > 0.5).astype(np.uint8)
        # Crop the mask to the same region as the displayed image
        mb_crop = mb[oy:oy+ch, ox:ox+cw]
        if mb_crop.shape != colored.shape[:2]: continue
        color = palette[i % len(palette)]
        ov = colored.copy()
        for c in range(3):
            ov[:,:,c] = np.where(mb_crop==1, color[c], ov[:,:,c])
        colored = cv2.addWeighted(colored, 0.3, ov, 0.7, 0)
        cnts,_ = cv2.findContours(mb_crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(colored, cnts, -1, color, 2)
    return colored

def compute_center(preds, shape):
    if len(preds["boxes"]) == 0:
        return shape[1]//2, shape[0]//2
    cy = float(np.median([(b[1]+b[3])/2 for b in preds["boxes"]]))
    cx = float(np.median([(b[0]+b[2])/2 for b in preds["boxes"]]))
    return int(cx), int(cy)

def assign_quadrants(preds, center):
    cx, cy = center
    quads = {"Q1":[], "Q2":[], "Q3":[], "Q4":[]}
    for i, box in enumerate(preds["boxes"]):
        tx, ty = (box[0]+box[2])/2, (box[1]+box[3])/2
        if ty < cy:
            q = "Q1" if tx < cx else "Q2"
        else:
            q = "Q3" if tx < cx else "Q4"
        quads[q].append({
            "idx":      i,
            "centroid": (int(tx), int(ty)),
            "box":      box,
            "label":    preds["labels"][i],
            "score":    preds["scores"][i],
            "mask":     preds["masks"][i],
        })
    # Sort within each quadrant from midline outwards
    for q, teeth in quads.items():
        quads[q] = sorted(teeth, key=lambda t: t["centroid"][0],
                          reverse=(q in ("Q1","Q3")))
    return quads

def number_teeth(quads):
    starts = {"Q1":11, "Q2":21, "Q3":31, "Q4":41}
    for q, teeth in quads.items():
        for rank, tooth in enumerate(teeth):
            tooth["number"]   = starts[q] + rank
            tooth["quadrant"] = q
    return quads

print("  All helpers defined: run_stage1, crop_to_teeth, color_teeth_fn,")
print("                      compute_center, assign_quadrants, number_teeth")
print()
print("CELL 8 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 9: Per-tooth Stage 2 inference helpers")
print("=" * 60)

# ================== CELL 9: PER-TOOTH ANOMALY DETECTION ==================

def crop_single_tooth(img_np, box, pad=CONFIG2["crop_pad"]):
    x1,y1,x2,y2 = box;  H,W = img_np.shape[:2]
    crop = img_np[
        max(0, int(y1)-pad) : min(H, int(y2)+pad),
        max(0, int(x1)-pad) : min(W, int(x2)+pad),
    ]
    return crop

def run_stage2_on_crop(crop_np):
    if crop_np.shape[0] < CONFIG2["min_crop_size"] or \
       crop_np.shape[1] < CONFIG2["min_crop_size"]:
        return []
    rgb = cv2.cvtColor(crop_np, cv2.COLOR_GRAY2RGB) if crop_np.ndim==2 else crop_np.copy()
    t   = TF.to_tensor(Image.fromarray(rgb)).to(device)
    stage2_model.eval()
    with torch.no_grad():
        pred = stage2_model([t])[0]
    results = []
    for box, lbl, score in zip(pred["boxes"].cpu().numpy(),
                                pred["labels"].cpu().numpy(),
                                pred["scores"].cpu().numpy()):
        if score < CONFIG2["conf_threshold"]: continue
        results.append({
            "label":      int(lbl),
            "label_name": ANOMALY_CLASSES.get(int(lbl), "Unknown"),
            "score":      float(score),
            "box":        box.tolist(),
        })
    return results

def detect_anomalies(img_np, numbered_quads):
    """Run Stage 2 on each individual tooth crop. Returns anomaly_map keyed by tooth number."""
    anomaly_map = {}
    total = sum(len(v) for v in numbered_quads.values())
    done  = 0
    for q_name, teeth in numbered_quads.items():
        for tooth in teeth:
            crop = crop_single_tooth(img_np, tooth["box"])
            dets = run_stage2_on_crop(crop)
            done += 1
            if dets:
                anomaly_map[tooth["number"]] = {
                    "quadrant": q_name,
                    "tooth":    tooth,
                    "anomalies":dets,
                }
    print(f"    Checked {done}/{total} teeth  →  {len(anomaly_map)} anomalies found")
    return anomaly_map

print("  crop_single_tooth  OK")
print("  run_stage2_on_crop OK")
print("  detect_anomalies   OK")
print()
print("CELL 9 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 10: Full pipeline + visualization (function definition)")
print("=" * 60)

# ================== CELL 10: FULL PIPELINE + VISUALIZATION ==================
# Produces a 4-panel figure identical to Stage 1 layout, with:
#  • quadrant divider lines
#  • per-tooth numbering (FDI notation)
#  • red bounding-box highlight + label for anomalous teeth
#  • anomaly report text panel

HIGHLIGHT_COLOR   = (255,  50,  50)   # red   – anomalous tooth outline
QUAD_LINE_COLOR   = (255, 230,   0)   # yellow quadrant dividers
NUM_NORMAL_COLOR  = (  0, 255, 255)   # cyan  – normal tooth number
NUM_ANOMALY_COLOR = (255,  50,  50)   # red   – anomalous tooth number


def run_full_pipeline(image_source, save_path=None):
    # ── Load image ───────────────────────────────────────────────────────────
    if isinstance(image_source, str):
        img_np = np.array(Image.open(image_source).convert("RGB"))
    elif isinstance(image_source, np.ndarray):
        img_np = (cv2.cvtColor(image_source, cv2.COLOR_GRAY2RGB)
                  if image_source.ndim==2 else image_source.copy())
    else:
        img_np = np.array(image_source.convert("RGB"))

    img_tensor = TF.to_tensor(Image.fromarray(img_np))

    # ── Stage 1 – teeth segmentation ────────────────────────────────────────
    print("  [1/4] Running Stage 1 teeth segmentation...")
    preds_s1       = run_stage1(img_tensor)
    n_teeth        = len(preds_s1["labels"])
    print(f"Stage 1 → {n_teeth} teeth detected")

    center         = compute_center(preds_s1, img_np.shape)
    quads          = assign_quadrants(preds_s1, center)
    numbered_quads = number_teeth(quads)
    cropped_np, crop_box = crop_to_teeth(img_np, preds_s1)
    colored_np     = color_teeth_fn(cropped_np.copy(), preds_s1, crop_box)

    # ── Stage 2 – per-tooth anomaly detection ───────────────────────────────
    print(f"  [2/4] Stage 1 detected {len(preds_s1['labels'])} teeth  |  Running Stage 2 per tooth...")
    anomaly_map = detect_anomalies(img_np, numbered_quads)
    print(f"Stage 2 → {len(anomaly_map)} anomalous teeth")
    for tnum, info in sorted(anomaly_map.items()):
        for a in info["anomalies"]:
            print(f"   Tooth {tnum:>2d} ({info['quadrant']}):  "
                  f"{a['label_name']}  conf={a['score']:.2f}")

    # ── Build annotated image ────────────────────────────────────────────────
    print("  [3/4] Building annotated 4-panel figure...")
    result_img = colored_np.copy()
    h, w = result_img.shape[:2]
    ox = crop_box[0] if crop_box else 0
    oy = crop_box[1] if crop_box else 0
    cx_c = center[0] - ox
    cy_c = center[1] - oy

    # Quadrant dividers
    cv2.line(result_img, (cx_c, 0),   (cx_c, h), QUAD_LINE_COLOR, 3)
    cv2.line(result_img, (0, cy_c),   (w,  cy_c), QUAD_LINE_COLOR, 3)

    # Quadrant labels
    for ql, pos in {"Q1":(10,40),"Q2":(w-80,40),"Q3":(10,h-20),"Q4":(w-80,h-20)}.items():
        cv2.putText(result_img, ql, pos, cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0,0,0),   5)
        cv2.putText(result_img, ql, pos, cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,255,255), 2)

    # Tooth numbers + anomaly highlights
    for q_name, teeth in numbered_quads.items():
        for tooth in teeth:
            tx = tooth["centroid"][0] - ox
            ty = tooth["centroid"][1] - oy
            if not (0 <= tx < w and 0 <= ty < h): continue
            num  = tooth["number"]
            is_a = num in anomaly_map
            if is_a:
                bx1 = max(0, int(tooth["box"][0]) - ox - 5)
                by1 = max(0, int(tooth["box"][1]) - oy - 5)
                bx2 = min(w, int(tooth["box"][2]) - ox + 5)
                by2 = min(h, int(tooth["box"][3]) - oy + 5)
                cv2.rectangle(result_img, (bx1,by1), (bx2,by2), HIGHLIGHT_COLOR, 6)
                cv2.rectangle(result_img, (bx1+4,by1+4), (bx2-4,by2-4), (255,150,0), 2)
                # Disease label above the box
                top_anomalies = sorted(anomaly_map[num]["anomalies"],
                                       key=lambda a: a["score"], reverse=True)
                lbl_text = top_anomalies[0]["label_name"]
                cv2.putText(result_img, lbl_text, (bx1, max(14,by1-8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0),     4)
                cv2.putText(result_img, lbl_text, (bx1, max(14,by1-8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,200,50), 1)
            nc = NUM_ANOMALY_COLOR if is_a else NUM_NORMAL_COLOR
            cv2.putText(result_img, str(num), (tx-20, ty+15),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,0,0), 5)
            cv2.putText(result_img, str(num), (tx-20, ty+15),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, nc, 2)

    # ── Anomaly report text ──────────────────────────────────────────────────
    report_lines = ["ANOMALY REPORT", "─" * 34]
    if anomaly_map:
        for tnum in sorted(anomaly_map):
            info = anomaly_map[tnum]
            for a in info["anomalies"]:
                report_lines.append(
                    f"  Tooth {tnum:>2d}  ({info['quadrant']}):  "
                    f"{a['label_name']}  [{a['score']*100:.0f}%]"
                )
    else:
        report_lines.append("  No anomalies detected.")
    report_text = "\n".join(report_lines)

    # ── 4-panel figure ───────────────────────────────────────────────────────
    fig = plt.figure(figsize=(22, 18), facecolor="#1a1a2e")
    gs  = fig.add_gridspec(2, 2, hspace=0.08, wspace=0.08,
                            left=0.02, right=0.98, top=0.93, bottom=0.22)
    tkw = dict(fontsize=14, fontweight="bold", color="white",
               fontfamily="monospace", pad=8)

    ax1 = fig.add_subplot(gs[0,0]); ax1.imshow(img_np, cmap="gray")
    ax1.set_title("Original Panoramic X-ray", **tkw); ax1.axis("off")

    ax2 = fig.add_subplot(gs[0,1]); ax2.imshow(cropped_np, cmap="gray")
    ax2.set_title("Cropped to Teeth Region  (Stage 1)", **tkw); ax2.axis("off")

    ax3 = fig.add_subplot(gs[1,0]); ax3.imshow(colored_np)
    ax3.set_title("Coloured Segmentation  (Stage 1)", **tkw); ax3.axis("off")

    ax4 = fig.add_subplot(gs[1,1]); ax4.imshow(result_img)
    ax4.set_title("Quadrants + Numbers + Anomaly Highlights  (Stage 1+2)", **tkw)
    ax4.axis("off")

    fig.suptitle("Dental X-ray Analysis Pipeline",
                 fontsize=20, fontweight="bold", color="white",
                 fontfamily="monospace", y=0.97)

    # Report box inside the figure at the bottom
    fig.text(0.02, 0.19, report_text,
             fontsize=10, color="#f0f0f0", fontfamily="monospace",
             va="top", transform=fig.transFigure,
             bbox=dict(boxstyle="round,pad=0.6",
                       facecolor="#0d0d1a",
                       edgecolor="#ff3232",
                       linewidth=1.5))

    np_norm = np.array(NUM_NORMAL_COLOR) / 255.
    np_anom = np.array(HIGHLIGHT_COLOR)  / 255.
    fig.legend(
        handles=[mpatches.Patch(color=np_norm, label="Normal tooth"),
                 mpatches.Patch(color=np_anom, label="Anomalous tooth")],
        loc="lower right", fontsize=11,
        facecolor="#1a1a2e", edgecolor="gray", labelcolor="white", framealpha=0.9,
    )

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        sz = os.path.getsize(save_path)
        print(f"  [4/4] Saved → {save_path}  ({sz/1024:.1f} KB)")
    plt.show()
    return fig, anomaly_map

print("  run_full_pipeline defined OK")
print()
print("CELL 10 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 11: Grad-CAM heatmaps")
print("=" * 60)

# ================== CELL 11: GRAD-CAM ON ANOMALOUS TEETH ==================
# Produces a heatmap overlay per anomalous tooth crop for interpretability.

class GradCAMFasterRCNN:
    """Simple Grad-CAM for Faster R-CNN backbone (layer4 of ResNet)."""
    def __init__(self, model):
        self.model    = model
        self.gradients = None
        self.activations = None
        target_layer = model.backbone.body.layer4
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, image_tensor, target_class=1):
        self.model.train()   # need gradients
        image_tensor = image_tensor.to(device).unsqueeze(0)
        outputs = self.model(image_tensor)
        preds   = outputs[0]
        if len(preds["scores"]) == 0:
            return None
        # Use the highest-confidence detection
        best_idx = preds["scores"].argmax()
        score    = preds["scores"][best_idx]
        score.backward(retain_graph=True)
        if self.gradients is None or self.activations is None:
            return None
        weights    = self.gradients.mean(dim=(2,3), keepdim=True)
        cam        = (weights * self.activations).sum(dim=1).squeeze()
        cam        = torch.clamp(cam, min=0)
        cam        -= cam.min();  cam /= (cam.max() + 1e-8)
        return cam.cpu().numpy()


def make_gradcam_overlay(crop_np, cam_np):
    h, w = crop_np.shape[:2]
    cam_resized = cv2.resize(cam_np, (w, h))
    heatmap     = cv2.applyColorMap((cam_resized*255).astype(np.uint8), cv2.COLORMAP_JET)
    rgb         = cv2.cvtColor(crop_np, cv2.COLOR_GRAY2BGR) if crop_np.ndim==2 else crop_np[:,:,::-1]
    overlay     = cv2.addWeighted(rgb, 0.5, heatmap, 0.5, 0)
    return overlay[:,:,::-1]   # back to RGB


print("  Initialising GradCAM...")
gradcam = GradCAMFasterRCNN(stage2_model)
print("  GradCAM ready")

# Pick the first DENTEX test image that has anomalies (or just the first image)
gradcam_img_path = None
for img_path, lbl, _ in test_ds.samples[:20]:
    if lbl is not None: gradcam_img_path = img_path; break

if gradcam_img_path is None:
    gradcam_img_path = test_ds.samples[0][0]
print(f"  Using: {os.path.basename(gradcam_img_path)}")

img_np_gc  = np.array(Image.open(gradcam_img_path).convert("RGB"))
img_tensor = TF.to_tensor(Image.fromarray(img_np_gc))
preds_gc   = run_stage1(img_tensor)
center_gc  = compute_center(preds_gc, img_np_gc.shape)
quads_gc   = assign_quadrants(preds_gc, center_gc)
numbered_gc = number_teeth(quads_gc)

# Run Grad-CAM on each anomalous tooth crop (up to 6 for display)
n_cols  = 3
anomaly_crops = []
for q_name, teeth in numbered_gc.items():
    for tooth in teeth:
        crop = crop_single_tooth(img_np_gc, tooth["box"])
        dets = run_stage2_on_crop(crop)
        if not dets: continue
        rgb   = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB) if crop.ndim==2 else crop.copy()
        t_crop = TF.to_tensor(Image.fromarray(rgb))
        cam   = gradcam.generate(t_crop, target_class=dets[0]["label"])
        if cam is None: continue
        overlay = make_gradcam_overlay(crop, cam)
        anomaly_crops.append({
            "tooth_num": tooth["number"],
            "quadrant":  q_name,
            "label":     dets[0]["label_name"],
            "score":     dets[0]["score"],
            "original":  crop if crop.ndim==3 else cv2.cvtColor(crop,cv2.COLOR_GRAY2RGB),
            "overlay":   overlay,
        })
        if len(anomaly_crops) >= 6: break
    if len(anomaly_crops) >= 6: break

stage2_model.eval()
print(f"  Grad-CAM generated for {len(anomaly_crops)} anomalous teeth")

if anomaly_crops:
    n_rows = len(anomaly_crops)
    fig, axes = plt.subplots(n_rows, 2, figsize=(10, 3.5*n_rows), facecolor="#1a1a2e")
    if n_rows == 1: axes = [axes]
    for row, ac in enumerate(anomaly_crops):
        axes[row][0].imshow(ac["original"]); axes[row][0].axis("off")
        axes[row][0].set_title(f"Tooth {ac['tooth_num']} ({ac['quadrant']}) – Crop",
                                color="white", fontsize=11)
        axes[row][1].imshow(ac["overlay"]);  axes[row][1].axis("off")
        axes[row][1].set_title(f"{ac['label']}  [{ac['score']*100:.0f}%] – Grad-CAM",
                                color="#ff9944", fontsize=11)
    fig.suptitle("Stage 2 – Grad-CAM Anomaly Heatmaps",
                 color="white", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR,"stage2_gradcam.png"), dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print("✓ Saved stage2_gradcam.png")
else:
    print("⚠ No anomalous teeth found in the sample – Grad-CAM skipped.")
    # Save a placeholder so the output file exists
    fig, ax = plt.subplots(figsize=(6,3), facecolor="#1a1a2e")
    ax.text(0.5,0.5,"No anomalous teeth detected in sample",
            ha="center",va="center",color="white",fontsize=13)
    ax.axis("off")
    plt.savefig(os.path.join(OUT_DIR,"stage2_gradcam.png"), dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
print()
print("CELL 11 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 12: Run full pipeline on 3 test images")
print("=" * 60)
import shutil

# ================== CELL 12: RUN FULL PIPELINE ON TEST SAMPLES ==================

test_imgs = sorted(
    glob.glob(os.path.join(DENTEX_TRAIN_IMG, "*.png"))[:3] +
    glob.glob(os.path.join(DENTEX_TRAIN_IMG, "*.jpg"))[:3]
)[:3]

prediction_figs = []
for i, img_path in enumerate(test_imgs):
    print(f"\n{'='*60}")
    print(f"Image {i+1}: {os.path.basename(img_path)}")
    print("="*60)
    save_p = os.path.join(OUT_DIR, f"stage2_predictions_{i+1}.png")
    fig, amap = run_full_pipeline(img_path, save_path=save_p)
    prediction_figs.append(fig)
    plt.close(fig)

# Rename first prediction to match expected output filename
first_out = os.path.join(OUT_DIR, "stage2_predictions_1.png")
if os.path.exists(first_out):
    shutil.copy(first_out, os.path.join(OUT_DIR, "stage2_predictions.png"))

print("\n✓ All outputs saved to /kaggle/working/")
print("  Files produced:")
for f in sorted(os.listdir(OUT_DIR)):
    if f.startswith("stage2"):
        sz = os.path.getsize(os.path.join(OUT_DIR, f))
        print(f"    {f}  ({sz/1024:.1f} KB)")
print()
print("CELL 12 COMPLETE ✓")


In [ ]:
print("=" * 60)
print("CELL 13: Custom image analysis")
print("=" * 60)

# ================== CELL 13: RUN ON CUSTOM IMAGE ==================
# Change IMAGE_PATH to any panoramic X-ray you want to analyse.

IMAGE_PATH = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/xrays/example.png"

if os.path.exists(IMAGE_PATH):
    fig, anomaly_map = run_full_pipeline(
        IMAGE_PATH,
        save_path=os.path.join(OUT_DIR, "stage2_analysis_custom.png")
    )
    plt.close(fig)
else:
    print(f"⚠ Image not found: {IMAGE_PATH}")
    print("  Edit IMAGE_PATH above to point to any panoramic X-ray .png/.jpg")

print()
print("CELL 13 COMPLETE ✓")
print()
print("=" * 60)
print("ALL CELLS DONE ✓")
print("=" * 60)
